Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.4 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import random

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    IA3Config,
    PrefixTuningConfig,
    get_peft_model
)
import evaluate


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


Model

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

Dataset (MRPC)

In [ ]:
task = "mrpc"
dataset = load_dataset("glue", task)

metric = evaluate.load("glue", task)
num_labels = 2


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Dataset helper

In [ ]:
def get_subset(dataset, fraction, seed=42):
    size = int(len(dataset) * fraction)
    return dataset.shuffle(seed=seed).select(range(size))

def build_datasets(fraction):
    small_train = get_subset(dataset["train"], fraction)

    train_ds = small_train.map(preprocess, batched=True)
    val_ds = dataset["validation"].map(preprocess, batched=True)

    keep_cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

    train_ds = train_ds.rename_column("label", "labels")
    val_ds = val_ds.rename_column("label", "labels")

    train_ds.set_format("torch")
    val_ds.set_format("torch")

    return train_ds, val_ds

Metrices

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

Trainable param

In [ ]:
def print_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

**Training arguments**

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

training_args_peft = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=5e-4,      # 🔑 MUCH higher
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,     # 🔑 MORE epochs
    weight_decay=0.0,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)


Train

In [ ]:
def train_and_eval(model, name, train_ds, val_ds):
    print(f"\n===== Training: {name} =====")
    print_trainable_params(model)

    args = training_args_full if name == "Full FT" else training_args_peft

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer.evaluate()


Model

In [ ]:
def build_models():
    # Full FT
    model_full = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )

    # BitFit
    model_bitfit = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )
    for name, param in model_bitfit.named_parameters():
        if "bias" not in name:
            param.requires_grad = False

    # LoRA
    lora_config = LoraConfig(
    r=16,                  # 🔑 more capacity
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    task_type="SEQ_CLS"
    )

    model_lora = get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels),
        lora_config
    )

    # IA3
    ia3_config = IA3Config(task_type="SEQ_CLS")
    model_ia3 = get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels),
        ia3_config
    )

    # Prefix
    prefix_config = PrefixTuningConfig(
        task_type="SEQ_CLS",
        num_virtual_tokens=20
    )
    model_prefix = get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels),
        prefix_config
    )

    return model_full, model_bitfit, model_lora, model_ia3, model_prefix

Dataset fraction

In [ ]:
fractions = [0.3, 0.5, 0.7, 1.0]
all_results = {}

for frac in fractions:
    print(f"\n===== Training with {int(frac*100)}% data =====")

    train_ds, val_ds = build_datasets(frac)
    model_full, model_bitfit, model_lora, model_ia3, model_prefix = build_models()

    results_frac = {}
    results_frac["Full FT"] = train_and_eval(model_full, "Full FT", train_ds, val_ds)
    results_frac["BitFit"]  = train_and_eval(model_bitfit, "BitFit", train_ds, val_ds)
    results_frac["LoRA"]    = train_and_eval(model_lora, "LoRA", train_ds, val_ds)
    results_frac["IA3"]     = train_and_eval(model_ia3, "IA3", train_ds, val_ds)
    results_frac["Prefix"] = train_and_eval(model_prefix, "Prefix", train_ds, val_ds)

    all_results[int(frac*100)] = results_frac

    del model_full, model_bitfit, model_lora, model_ia3, model_prefix
    torch.cuda.empty_cache()



===== Training with 30% data =====


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.507040,0.436548,0.801471,0.866116
2,0.378499,0.365004,0.835784,0.888889
3,0.243840,0.336295,0.867647,0.903571
4,0.236862,0.362038,0.862745,0.901754
5,0.128932,0.555672,0.850490,0.891266
6,0.059830,0.585965,0.857843,0.896797
7,0.107323,0.633643,0.848039,0.884328
8,0.060634,0.802309,0.860294,0.899824
9,0.044753,0.750798,0.865196,0.902998
10,0.048523,0.782862,0.843137,0.888889



===== Training: IA3 =====
Trainable params: 66,050 / 109,549,828 (0.0603%)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.640513,0.632986,0.681373,0.810496
2,0.579126,0.560313,0.718137,0.821151
3,0.516229,0.640725,0.705882,0.821429
4,0.532536,0.523400,0.764706,0.842623
5,0.532514,0.521414,0.769608,0.848875
6,0.493018,0.502888,0.762255,0.831304
7,0.490136,0.508014,0.767157,0.844517
8,0.482391,0.575103,0.776961,0.857143
9,0.453563,0.525601,0.769608,0.849359
10,0.431504,0.522797,0.772059,0.851200



===== Training: Prefix =====
Trainable params: 370,178 / 109,853,956 (0.3370%)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.620735,0.637785,0.686275,0.813411
2,0.616281,0.589318,0.691176,0.810241
3,0.569742,0.656456,0.686275,0.811209
4,0.599901,0.566941,0.718137,0.822257
5,0.577334,0.557594,0.720588,0.820755
6,0.572360,0.565748,0.703431,0.791019
7,0.566733,0.547200,0.737745,0.828250
8,0.567369,0.594924,0.718137,0.826021
9,0.546821,0.558749,0.730392,0.829721
10,0.512826,0.545799,0.730392,0.825949
